In [1]:
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
import geopandas as gpd
from libpysal import weights
from sklearn.preprocessing import LabelEncoder
import networkx as nx
import re
import os
import pickle
from scipy.sparse import diags

/home/yuyue/stan_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
strat_df = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_sf.csv")

In [3]:
df = pd.read_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/preprocessed_cleaned_all_result.pkl')

In [4]:
print(df.columns.to_list())

['id', 'time_window_start', 'tweets_struct', 'name', 'username', 'description', 'location', 'public_metrics.followers_count', 'profile_image_url', 'clean...pool', 'is_empty_window', 'window_weeks_to_election', 'profile_image_description', 'profile_image_main_info', 'Model', 'Belief_Raw', 'Belief_Cleaned', 'Demo_Raw', 'Demo_Cleaned', 'Belief_opinion_on_the_number_of_immigrants_in_america_Category', 'Belief_opinion_on_the_number_of_immigrants_in_america_Explanation', 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol', 'Belief_opinion_on_the_number_of_immigrants_in_america_Speculation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Category', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Explanation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Symbol', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Speculation', 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_t

In [5]:
belief_col = "Belief_opinion_on_what_should_be_done_with_illegal_immigrants_Symbol"

In [6]:
predictor_cols = [
    "Demo_sex_Symbol",
    "Demo_age_Symbol",
    "Demo_ethnicity_Symbol",
    "Demo_income_Symbol",
    "Demo_past_vote_Symbol",
    "state_cleaned"
]
df = df.dropna(subset=[belief_col] + predictor_cols)

In [7]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_opinion_on_whether_the_tone_of_political_discourse_in_the_u_s_has_become_more_factbased_Symbol_cat,Belief_opinion_on_whether_the_tone_of_political_discourse_in_the_u_s_has_become_more_factbased_Symbol_num,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_cat,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_num,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_past_vote_idx,state_cleaned_idx
40,1542629834953719810,2024-05-19 00:00:00+00:00,"['24 weeks to election', {'text': '@The_Trump_...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dic2,2,dib1,1,2.0,3.0,1.0,3.0,2.0,36.0
41,1542629834953719810,2024-06-16 00:00:00+00:00,"['20 weeks to election', {'text': '🤣 https://t...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dic2,2,dib1,1,2.0,3.0,1.0,3.0,2.0,36.0
43,1542629834953719810,2024-07-14 00:00:00+00:00,"['16 weeks to election', {'text': '🤣🤣🤣 https:/...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dic2,2,dib3,3,2.0,3.0,1.0,3.0,2.0,36.0
44,1542629834953719810,2024-07-28 00:00:00+00:00,"['14 weeks to election', {'text': '@victormarx...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dic2,2,dib1,1,2.0,3.0,1.0,2.0,3.0,36.0
45,1542629834953719810,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '🤣🥰 https://...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dic2,2,dib1,1,2.0,3.0,1.0,3.0,2.0,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dic2,2,dib1,1,1.0,2.0,1.0,4.0,3.0,15.0
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dic2,2,dib1,1,1.0,2.0,1.0,3.0,3.0,15.0
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dic2,2,dib1,1,1.0,2.0,1.0,3.0,3.0,15.0
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dic2,2,dib1,1,1.0,2.0,1.0,3.0,1.0,15.0


In [8]:
# prepare spatial structure for ICAR
def edges_from_adjacency(W):
    all_edges = []
    n = W.shape[0]
    
    for i in range(n):
        neighbors = np.where(W[i, :] != 0)[0]
        for j in neighbors:
            if i < j:  
                all_edges.append({
                    'node1': i + 1,  
                    'node2': j + 1,
                    'weight': W[i, j]
                })
    
    return pd.DataFrame(all_edges)


def scale_c(C):
    N = C.shape[0]
    
    if hasattr(C, 'toarray'):
        C = C.toarray()
    C = np.asarray(C)
    
    row_sums = C.sum(axis=1).ravel()
    Q = diags(row_sums) - C
    
    Q_dense = Q.toarray() if hasattr(Q, 'toarray') else np.asarray(Q)
    max_diag = np.max(np.diag(Q_dense))
    jitter = max_diag * np.sqrt(np.finfo(float).eps)
    Q_pert = Q_dense + np.diag(np.ones(N) * jitter)
    
    Q_inv = np.linalg.pinv(Q_pert)
    
    variances = np.diag(Q_inv)
    scaling_factor = np.exp(np.mean(np.log(variances)))
    
    return scaling_factor


def prep_icar_data_from_weights(w, N_states):
    C = np.zeros((N_states, N_states))
    for i in range(N_states):
        for j in w.neighbors[i]:
            C[i, j] = 1

    E = edges_from_adjacency(C)
    
    G = nx.Graph()
    G.add_nodes_from(range(N_states))
    for _, row in E.iterrows():
        G.add_edge(int(row['node1']) - 1, int(row['node2']) - 1)
    
    components = list(nx.connected_components(G))
    k = len(components)
    
    comp_id = np.zeros(N_states, dtype=int)
    for idx, component in enumerate(components):
        for node in component:
            comp_id[int(node)] = idx
    
    group_size = np.array([len(comp) for comp in components])
    
    group_idx = comp_id + 1

    
    return {
        'k': int(k),
        'group_size': group_size.tolist(),
        'N_edges': len(E),
        'node1': E['node1'].values.tolist(),
        'node2': E['node2'].values.tolist(),
        'group_idx': group_idx.tolist()
    }


def moran_coefficient(x, W, warn=True):
    row_sums = W.sum(axis=1)
    zero_idx = np.where(row_sums == 0)[0]
    
    if len(zero_idx) > 0:
        if warn:
            print(f"{len(zero_idx)} observations with no neighbors. Dropping from calculation.")
        keep_idx = row_sums > 0
        x = x[keep_idx]
        W = W[np.ix_(keep_idx, keep_idx)]
    
    n = len(x)
    xbar = np.mean(x)
    z = x - xbar
    z_tilde = W @ z
    A = np.sum(W)
    mc = (n / A) * (z @ z_tilde) / (z @ z)
    
    return np.round(mc, 3)

# Build spatial structure
shp = gpd.read_file("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/tl_2025_us_state/tl_2025_us_state.shp")
shp["STATE"] = shp["STUSPS"].str.upper()

shp = shp[~shp["STATE"].isin(["PR", "GU", "VI", "MP", "AS"])]

shp = shp.sort_values("STATE").reset_index(drop=True)

state_order = shp["STATE"].tolist()
N_states = len(state_order)

print(f"Shapefile states: {N_states}")

w = weights.Queen.from_dataframe(shp)

icar_data = prep_icar_data_from_weights(w, N_states)

print(f"\nICAR structure:")
print(f"  Connected components: {icar_data['k']}")
print(f"  Component sizes: {icar_data['group_size']}")
print(f"  Number of edges: {icar_data['N_edges']}")



Shapefile states: 51


/tmp/ipykernel_2478/4196857512.py:111: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(shp)



ICAR structure:
  Connected components: 3
  Component sizes: [1, 49, 1]
  Number of edges: 112


/home/yuyue/stan_env/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There are 2 islands with ids: 0, 11.
  W.__init__(self, neighbors, ids=ids, **kw)


In [9]:
shp.head()

,REGION,DIVISION,STATEFP,STATENS,GEOID,GEOIDFQ,STUSPS,NAME,LSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,STATE
0,4,9,02,01785533,02,0400000US02,AK,Alaska,00,G4000,A,1479893380150,244326118163,+63.3473560,-152.8397334,"MULTIPOLYGON (((-167.55823 60.22436, -167.5567...",AK
1,3,6,01,01779775,01,0400000US01,AL,Alabama,00,G4000,A,131186429591,4580729056,+32.7395785,-086.8434469,"POLYGON ((-85.12733 31.76256, -85.12753 31.762...",AL
2,3,7,05,00068085,05,0400000US05,AR,Arkansas,00,G4000,A,134658642291,3122591253,+34.8955256,-092.4446262,"POLYGON ((-90.95577 34.11871, -90.95451 34.117...",AR
3,4,8,04,01779777,04,0400000US04,AZ,Arizona,00,G4000,A,294366238828,853868947,+34.2039362,-111.6063449,"POLYGON ((-110.75069 37.00301, -110.74193 37.0...",AZ
4,4,9,06,01779778,06,0400000US06,CA,California,00,G4000,A,403676564152,20288505023,+37.1551773,-119.5434183,"MULTIPOLYGON (((-119.99987 41.18397, -119.9998...",CA


In [3]:
# state level fixed effects
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_sf.csv")
LP_cols = [
    'LP_pct.2020.votes.R', 'LP_pct.2020.votes.stay.home'
]
LP_state = (
    strat_frame
    .assign(STATE=strat_frame["state_abbreviation"].str.upper())
    .groupby("STATE")[LP_cols]
    .mean()
)
LP_state = LP_state.reindex(state_order)

LP_mat = LP_state.values
P = LP_mat.shape[1]
print(LP_mat.shape)

NameError: name 'state_order' is not defined

In [4]:
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_sf.csv")

In [5]:
strat_frame

,Unnamed: 0,sex_idx,age_idx,eth_idx,income_idx,state_idx,vote_idx,N_star.1,N_star.2,N_star.3,...,LP_pct.2020.votes.G,LP_pct.2020.votes.stay.home,LP_pct.2020.votes.OTHER,LP_pct.2022.votes.D,LP_pct.2022.votes.R,LP_pct.2022.votes.L,LP_pct.2022.votes.G,LP_pct.2022.votes.stay.home,LP_pct.2022.votes.OTHER,state_abbreviation_upper
0,0,2,1,4,1,1,1,1.867455,0.548105,1.565072,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
1,1,2,1,4,1,1,2,0.013752,0.139760,0.207898,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
2,2,2,1,4,1,1,3,0.264992,1.602773,0.317292,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
3,3,2,1,4,1,1,4,0.001838,0.007018,0.013940,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
4,4,2,1,4,1,1,5,0.000000,0.000000,0.000000,...,-0.997116,-0.154947,2.071398,0.286556,-0.196120,0.136231,-0.175703,-0.514353,-0.467282,AK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110155,110155,1,6,1,5,51,2,23.309498,26.801340,33.842180,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY
110156,110156,1,6,1,5,51,3,17.174077,19.340563,9.853957,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY
110157,110157,1,6,1,5,51,4,0.102195,0.023442,0.284320,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY
110158,110158,1,6,1,5,51,5,0.000000,0.000000,0.000000,...,-0.997116,0.314307,2.264716,-1.463526,1.104902,0.441713,-0.175703,0.042198,1.215613,WY


In [2]:
import pandas as pd

In [11]:
print(LP_mat)

[[ 0.3032589  -0.15494699]
 [ 1.06868261  0.92695691]
 [ 1.09897892  1.87393581]
 [-0.01104556 -0.13549883]
 [-1.23719282 -0.06126084]
 [-0.63112784 -1.72932917]
 [-0.83225137 -0.36399364]
 [-3.64396112  0.30521813]
 [-0.78334293 -0.39551991]
 [ 0.16901134 -0.80350951]
 [ 0.00405679 -0.12421476]
 [-1.26764912  1.97166449]
 [ 0.30051743 -0.98120931]
 [ 1.2189566   0.05901784]
 [-0.71855639  0.2087094 ]
 [ 0.65175111  1.08771718]
 [ 0.58446232  0.28206992]
 [ 1.07325333  0.47984469]
 [ 0.77155319  0.61767264]
 [-1.43794678 -0.65951938]
 [-1.41779837 -0.34927793]
 [-0.41779583 -1.474638  ]
 [-0.11245498 -1.00240798]
 [-0.32484046 -2.09808463]
 [ 0.63331954  0.20469241]
 [ 0.70016347  1.25593282]
 [ 0.64320622 -0.79548343]
 [ 0.06204364 -0.91829113]
 [ 1.32517512  0.71970753]
 [ 0.83397673 -0.2725791 ]
 [-0.31890604 -1.20076828]
 [-0.64840276 -0.5917758 ]
 [-0.47364589  1.10262997]
 [-0.1266878   0.19441404]
 [-0.97586033  1.07755046]
 [ 0.33971872  0.12281927]
 [ 1.34674554  2.06296573]
 

In [12]:
LP_df = pd.DataFrame(
    LP_mat,
    index=[s.upper() for s in state_order],
    columns=LP_cols
)

LP_df

,LP_pct.2020.votes.R,LP_pct.2020.votes.stay.home
AK,0.303259,-0.154947
AL,1.068683,0.926957
AR,1.098979,1.873936
AZ,-0.011046,-0.135499
CA,-1.237193,-0.061261
CO,-0.631128,-1.729329
CT,-0.832251,-0.363994
DC,-3.643961,0.305218
DE,-0.783343,-0.395520
FL,0.169011,-0.803510


In [13]:
print(df["Demo_age_idx"].unique())

[3. 4. 5. 2. 6. 1.]


In [14]:
# Stan code
stan_code = '''
functions {
  real icar_normal_lpdf(vector phi, int N_edges,
                        array[] int node1, array[] int node2) {
    return -0.5 * dot_self(phi[node1] - phi[node2]);
  }
}

data {
  int<lower=1> N;                   
  int<lower=2> K;                 
  array[N] int<lower=1, upper=K> y;

  int<lower=1> J_sex;
  int<lower=1> J_age;
  int<lower=1> J_eth;
  int<lower=1> J_income;
  int<lower=1> J_vote;
  int<lower=1> N_states;

  array[N] int<lower=1, upper=J_sex> sex_idx;
  array[N] int<lower=1, upper=J_age> age_idx;
  array[N] int<lower=1, upper=J_eth> eth_idx;
  array[N] int<lower=1, upper=J_income> income_idx;
  array[N] int<lower=1, upper=J_vote> vote_idx;
  array[N] int<lower=1, upper=N_states> state_idx;

  // ICAR structure
  int<lower=1> N_edges;
  array[N_edges] int node1;
  array[N_edges] int node2;

  int<lower=1> k;                           // number of connected components
  array[k] int group_size;                  // size of each connected component
  array[N_states] int group_idx;           // component assignment for each state

  // state level fixed effects
  int<lower=1> P;                           // number of state-level predictors
  matrix[N_states, P] LP;                  // state-level predictor matrix
}

parameters {

  matrix[J_sex, K-1] eta_sex; // slopes for each predictor category x outcome category - 1
  matrix[J_age, K-1] eta_age;             
  matrix[J_eth, K-1] eta_eth;
  matrix[J_income, K-1] eta_income;      
  matrix[J_vote, K-1] eta_vote;
  matrix[N_states, K-1] eta_state_raw;
  matrix[P,K-1] beta_LP;
  vector[K-1] alpha; // intercepts

  real<lower=0> sigma_sex;
  real<lower=0> sigma_age;
  real<lower=0> sigma_eth;
  real<lower=0> sigma_income;
  real<lower=0> sigma_vote;
  real<lower=0> sigma_state;
}

transformed parameters {
  matrix[J_sex, K-1] eta_sex_star = eta_sex * sigma_sex * sqrt((J_sex / (J_sex - 1.0)));
  matrix[J_age, K-1] eta_age_star = eta_age * sigma_age;
  matrix[J_eth, K-1] eta_eth_star = eta_eth * sigma_eth * sqrt((J_eth / (J_eth - 1.0)));
  matrix[J_income, K-1] eta_income_star = eta_income * sigma_income;
  matrix[J_vote, K-1] eta_vote_star = eta_vote * sigma_vote * sqrt((J_vote / (J_vote - 1.0)));
  matrix[N_states, K-1] mu_state_LP = LP * beta_LP; 
  matrix[N_states, K-1] eta_state = sigma_state * eta_state_raw;
}

model {
  // Priors
  alpha ~ normal(0, 5);

   // prior on the constrained vector (Stan handles the internal K-1 representation)
  to_vector(eta_sex) ~ normal(0, 1);
  to_vector(eta_eth) ~ normal(0, 1);
  to_vector(eta_vote) ~ normal(0, 1);
  
  sigma_sex ~ normal(0, 1);
  sigma_age ~ normal(0, 1);
  sigma_eth ~ normal(0, 1);
  sigma_income ~ normal(0, 1);
  sigma_vote ~ normal(0, 1);
  sigma_state ~ normal(0, 1);

  // random walk prior
  for (j in 1:(K-1)){ //loop through each output category
    // sum to zero constraint
    sum(eta_age[,j]) ~ normal(0, 0.01 * J_age);
    for (i in 2:J_age) {
      eta_age[i, j] ~ normal(eta_age[i-1,j], 1);
    }

  }
  for (j in 1:(K-1)){ 
    // sum to zero constraint
    sum(eta_income[,j]) ~ normal(0, 0.01 * J_income);
      for (i in 2:J_income) {
      eta_income[i, j] ~ normal(eta_income[i-1,j], 1);
      }
    }

  // ICAR prior on raw spatial effects
  for (c in 1:(K-1)){ 
    target += icar_normal_lpdf(eta_state_raw[, c] | N_edges, node1, node2);
    
    // Sum-to-zero constraint per connected component (island can have free prior)
    for (j in 1:k) {
    if (group_size[j] > 1) {
      // Soft sum-to-zero constraint for proper ICAR within this component
      real component_sum = 0;
      for (i in 1:N_states) {
        if (group_idx[i] == j) {
          component_sum += eta_state_raw[i,c];
        }
      }
      target += normal_lpdf(component_sum | 0, 0.001 * group_size[j]);
    } else {
      // Island component: give the singleton node a proper iid prior
      for (i in 1:N_states) {
        if (group_idx[i] == j) {
          eta_state_raw[i,c] ~ normal(0, 1);
        }
      }
    }
  }
}
  // Multinomial logistic
  for (i in 1:N) {
    vector[K] eta_i; // linear predictor for observation i
    // outcome category 1 to K-1
    for (s in 1:(K-1)) {
      eta_i[s] = alpha[s] 
                + eta_sex_star[sex_idx[i], s]
                + eta_age_star[age_idx[i], s]
                + eta_eth_star[eth_idx[i], s]
                + eta_income_star[income_idx[i], s]
                + eta_vote_star[vote_idx[i], s]
                + eta_state[state_idx[i], s]
                + mu_state_LP[state_idx[i], s];
  }
    // baseline category K
    eta_i[K] = 0;
    y[i] ~ categorical_logit(eta_i);
}
}
generated quantities {
  matrix[N,K] y_prob;
  for (i in 1:N) {
    vector[K] eta_i; // linear predictor for observation i
    for (s in 1:(K-1)) {
      eta_i[s] = alpha[s]
                + eta_sex_star[sex_idx[i], s]
                + eta_age_star[age_idx[i], s]
                + eta_eth_star[eth_idx[i], s]
                + eta_income_star[income_idx[i], s]
                + eta_vote_star[vote_idx[i], s]
                + eta_state[state_idx[i], s]
                + mu_state_LP[state_idx[i], s];
    }
    eta_i[K] = 0;
    y_prob[i] = softmax(eta_i)';
  }
}
'''


# Compile and Sample

with open('mrp.stan', 'w') as f:
    f.write(stan_code)

model = CmdStanModel(stan_file='mrp.stan')


00:45:41 - cmdstanpy - INFO - compiling stan file /tmp/tmpj5ayi9v2/tmpg9k5r4lu.stan to exe file /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp
00:45:57 - cmdstanpy - INFO - compiled model executable: /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp


In [15]:
time_windows = sorted(df['time_window_start'].unique())
print(f"Time windows found: {time_windows}")

for idx,window in enumerate(time_windows):
    df_window = df[df['time_window_start'] == window].copy()
    print(len(df_window))

Time windows found: ['2023-11-05 00:00:00+00:00', '2023-11-19 00:00:00+00:00', '2023-12-03 00:00:00+00:00', '2023-12-17 00:00:00+00:00', '2023-12-31 00:00:00+00:00', '2024-01-14 00:00:00+00:00', '2024-01-28 00:00:00+00:00', '2024-02-11 00:00:00+00:00', '2024-02-25 00:00:00+00:00', '2024-03-10 00:00:00+00:00', '2024-03-24 00:00:00+00:00', '2024-04-07 00:00:00+00:00', '2024-04-21 00:00:00+00:00', '2024-05-05 00:00:00+00:00', '2024-05-19 00:00:00+00:00', '2024-06-02 00:00:00+00:00', '2024-06-16 00:00:00+00:00', '2024-06-30 00:00:00+00:00', '2024-07-14 00:00:00+00:00', '2024-07-28 00:00:00+00:00', '2024-08-11 00:00:00+00:00', '2024-08-25 00:00:00+00:00', '2024-09-08 00:00:00+00:00', '2024-09-22 00:00:00+00:00', '2024-10-06 00:00:00+00:00', '2024-10-20 00:00:00+00:00']
165
166
162
177
184
195
203
209
220
235
245
251
259
269
311
307
329
351
397
417
439
480
541
421
300
123


In [ ]:
time_windows = sorted(df['time_window_start'].unique())
print(f"Time windows found: {time_windows}")
print(len(time_windows))

results_by_window = {}

for idx,window in enumerate(time_windows):
    # Filter data for this window
    print(f"[{idx+1}/{len(time_windows)}] Processing window: {window}")
    df_window = df[df['time_window_start'] == window].copy()
    print(f"  Observations: {len(df_window)}")
    print(f"  Belief categories: {sorted(df_window[belief_col+'_num'].unique())}")
    stan_data = {
        "N": len(df_window),
        "K": df[belief_col+"_cat"].cat.categories.size,
        "y": df_window[belief_col+"_num"].astype(int).tolist(),
        
        # Predictor dimensions
        "J_sex": strat_df['sex_idx'].nunique(),
        "J_age": strat_df['age_idx'].nunique(),
        "J_eth": strat_df['eth_idx'].nunique(),
        "J_income": strat_df['income_idx'].nunique(),
        "J_vote": strat_df['vote_idx'].nunique(),
        "N_states": N_states,
        

        # Predictor indices
        "sex_idx": df_window['Demo_sex_idx'].astype(int).tolist(),
        "age_idx": df_window['Demo_age_idx'].astype(int).tolist(),
        "eth_idx": df_window['Demo_ethnicity_idx'].astype(int).tolist(),
        "income_idx": df_window['Demo_income_idx'].astype(int).tolist(),
        "vote_idx": df_window['Demo_past_vote_idx'].astype(int).tolist(),
        "state_idx": df_window['state_cleaned_idx'].astype(int).tolist(),

        # ICAR spatial structure
        **icar_data,

        # state level fix effect
        "P": LP_mat.shape[1],
        "LP": LP_mat.tolist(),
    }
    
    fit = model.sample(
        data=stan_data,
        chains=8,
        parallel_chains=8,
        iter_warmup=2000,
        iter_sampling=1000,
        thin = 10,
        adapt_delta=0.995,
        max_treedepth=13,
        sig_figs=18,
        show_console=True
    )

    print(fit.summary())

    base_dir = "/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Nominal_Immigrants"
    model_dir = f"{base_dir}/{belief_col}_{window}_stan_output"

    os.makedirs(model_dir, exist_ok=True)

    fit.save_csvfiles(dir=f"{model_dir}/mrp_results/")
    summary_df = fit.summary()

    summary_df.to_csv(f"{model_dir}/mrp_summary.csv", index=True)

    posterior_draws = fit.draws_pd()  

    posterior_draws.to_csv(f"{model_dir}/mrp_draws.csv", index=False)
    with open(f"{model_dir}/fit.pkl", "wb") as f:
        pickle.dump(fit, f)
        

00:46:04 - cmdstanpy - INFO - Chain [1] start processing
00:46:04 - cmdstanpy - INFO - Chain [2] start processing
00:46:04 - cmdstanpy - INFO - Chain [3] start processing
00:46:04 - cmdstanpy - INFO - Chain [4] start processing
00:46:04 - cmdstanpy - INFO - Chain [5] start processing
00:46:04 - cmdstanpy - INFO - Chain [6] start processing
00:46:04 - cmdstanpy - INFO - Chain [7] start processing
00:46:04 - cmdstanpy - INFO - Chain [8] start processing


Time windows found: ['2023-11-05 00:00:00+00:00', '2023-11-19 00:00:00+00:00', '2023-12-03 00:00:00+00:00', '2023-12-17 00:00:00+00:00', '2023-12-31 00:00:00+00:00', '2024-01-14 00:00:00+00:00', '2024-01-28 00:00:00+00:00', '2024-02-11 00:00:00+00:00', '2024-02-25 00:00:00+00:00', '2024-03-10 00:00:00+00:00', '2024-03-24 00:00:00+00:00', '2024-04-07 00:00:00+00:00', '2024-04-21 00:00:00+00:00', '2024-05-05 00:00:00+00:00', '2024-05-19 00:00:00+00:00', '2024-06-02 00:00:00+00:00', '2024-06-16 00:00:00+00:00', '2024-06-30 00:00:00+00:00', '2024-07-14 00:00:00+00:00', '2024-07-28 00:00:00+00:00', '2024-08-11 00:00:00+00:00', '2024-08-25 00:00:00+00:00', '2024-09-08 00:00:00+00:00', '2024-09-22 00:00:00+00:00', '2024-10-06 00:00:00+00:00', '2024-10-20 00:00:00+00:00']
26
[1/26] Processing window: 2023-11-05 00:00:00+00:00
  Observations: 165
  Belief categories: [np.int8(1), np.int8(2), np.int8(3)]
Chain [6] method = sample (Default)
Chain [6] sample
Chain [6] num_samples = 1000 (Default)


00:50:18 - cmdstanpy - INFO - Chain [1] done processing


Chain [1] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [1] 
Chain [1] Elapsed Time: 180.593 seconds (Warm-up)
Chain [1] 73.356 seconds (Sampling)
Chain [1] 253.949 seconds (Total)
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [5] Iteration: 2900 / 3000 [ 96%]  (Sampling)


00:50:19 - cmdstanpy - INFO - Chain [4] done processing


Chain [4] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [4] 
Chain [4] Elapsed Time: 180.283 seconds (Warm-up)
Chain [4] 74.258 seconds (Sampling)
Chain [4] 254.541 seconds (Total)
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 


... (456 outputs left out)

04:12:28 - cmdstanpy - INFO - Chain [2] done processing


Chain [2] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [2] 
Chain [2] Elapsed Time: 144.006 seconds (Warm-up)
Chain [2] 98.878 seconds (Sampling)
Chain [2] 242.884 seconds (Total)
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 
Chain [2] 


04:12:29 - cmdstanpy - INFO - Chain [1] done processing
04:12:29 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: categorical_logit_lpmf: log odds parameter[1] is -inf, but must be finite! (in 'mrp.stan', line 145, column 4 to column 36)
Exception: categorical_logit_lpmf: log odds parameter[1] is -inf, but must be finite! (in 'mrp.stan', line 145, column 4 to column 36)
Exception: categorical_logit_lpmf: log odds parameter[1] is -inf, but must be finite! (in 'mrp.stan', line 145, column 4 to column 36)
Exception: categorical_logit_lpmf: log odds parameter[1] is inf, but must be finite! (in 'mrp.stan', line 145, column 4 to column 36)


Chain [1] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [1] 
Chain [1] Elapsed Time: 143.468 seconds (Warm-up)
Chain [1] 100.806 seconds (Sampling)
Chain [1] 244.274 seconds (Total)
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Ch

# Get Predictions

In [17]:
import pandas as pd
import re

def generate_cell_level_predictions(
    fit,
    strat_frame,
    belief_col,
    batch_size=10000,
    n_keep_draws=500
):

    # Extract posterior samples (multinomial logit)
    alpha = fit.stan_variable('alpha')  # (draws x K-1)
    eta_sex_star = fit.stan_variable('eta_sex_star')  # (draws x J_sex x K-1)
    eta_age_star = fit.stan_variable('eta_age_star')  # (draws x J_age x K-1)
    eta_eth_star = fit.stan_variable('eta_eth_star')  # (draws x J_eth x K-1)
    eta_vote_star = fit.stan_variable('eta_vote_star')  # (draws x J_vote x K-1)
    eta_income_star = fit.stan_variable('eta_income_star')  # (draws x J_income x K-1)
    eta_state = fit.stan_variable('eta_state')  # (draws x N_states x K-1)
    mu_state_LP = fit.stan_variable('mu_state_LP')  # (draws x N_states x K-1)

    n_draws_total = alpha.shape[0]
    n_cells = len(strat_frame)
    K = alpha.shape[1] + 1
    
    # Randomly select indices for draw subsampling
    draw_indices = np.random.choice(n_draws_total, size=min(n_keep_draws, n_draws_total), replace=False)
    print(f"Total draws available: {n_draws_total}, selected {len(draw_indices)} for subsampling")
    draw_indices = np.sort(draw_indices)  
    n_draws = len(draw_indices)
    
    # Subset posterior samples to keep only selected draws
    alpha = alpha[draw_indices, :]
    eta_sex_star = eta_sex_star[draw_indices, :, :]
    eta_age_star = eta_age_star[draw_indices, :, :]
    eta_eth_star = eta_eth_star[draw_indices, :, :]
    eta_vote_star = eta_vote_star[draw_indices, :, :]
    eta_income_star = eta_income_star[draw_indices, :, :]
    eta_state = eta_state[draw_indices, :, :]
    mu_state_LP = mu_state_LP[draw_indices, :, :]
    
    print(f"Subsampling draws: using {n_draws} out of {n_draws_total} posterior draws")

    # Initialize arrays
    pred_mean = np.zeros(n_cells)
    pred_median = np.zeros(n_cells)
    pred_lower = np.zeros(n_cells)
    pred_upper = np.zeros(n_cells)
    pred_sd = np.zeros(n_cells)
    prob_cats_mean = np.zeros((n_cells, K))
    prob_cats_median = np.zeros((n_cells, K))
    prob_cats_lower = np.zeros((n_cells, K))
    prob_cats_upper = np.zeros((n_cells, K))
    prob_cats_sd = np.zeros((n_cells, K))

    all_draws = np.zeros((n_cells, n_draws,))  # expected value draws
    mu_all_draws = np.zeros((n_cells, n_draws,))  # linear predictor draws for category 1

    n_batches = int(np.ceil(n_cells / batch_size))
    print(f"\nGenerating predictions for {n_cells:,} cells in {n_batches} batches of size {batch_size}")

    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, n_cells)
        batch_n = end_idx - start_idx
        batch_data = strat_frame.iloc[start_idx:end_idx]

        # logits for all categories (K) per draw
        logits_batch = np.zeros((batch_n, n_draws, K))

        for i, (_, row) in enumerate(batch_data.iterrows()):
            sex_i = int(row['sex_idx']) - 1
            age_i = int(row['age_idx']) - 1
            eth_i = int(row['eth_idx']) - 1
            income_i = int(row['income_idx']) - 1
            vote_i = int(row['vote_idx']) - 1
            state_i = int(row['state_idx']) - 1

            eta_i = (
                alpha
                + eta_sex_star[:, sex_i, :]
                + eta_age_star[:, age_i, :]
                + eta_vote_star[:, vote_i, :]
                + eta_eth_star[:, eth_i, :]
                + eta_income_star[:, income_i, :]
                + eta_state[:, state_i, :]
                + mu_state_LP[:, state_i, :]
            )

            logits_batch[i, :, :K-1] = eta_i
            logits_batch[i, :, K-1] = 0.0  # baseline category
            mu_all_draws[start_idx + i, :] = eta_i[:, 0]

        # softmax to probabilities (stable)
        max_logits = np.max(logits_batch, axis=2, keepdims=True)
        exp_logits = np.exp(logits_batch - max_logits)
        prob_batch = exp_logits / np.sum(exp_logits, axis=2, keepdims=True)

        # calculate expected value
        categories = np.arange(1, K+1)
        expected_y_batch = np.sum(prob_batch * categories[np.newaxis, np.newaxis, :], axis=2)

        # summary stats for expected value
        pred_mean[start_idx:end_idx] = np.mean(expected_y_batch, axis=1)
        pred_median[start_idx:end_idx] = np.median(expected_y_batch, axis=1)
        pred_lower[start_idx:end_idx] = np.percentile(expected_y_batch, 2.5, axis=1)
        pred_upper[start_idx:end_idx] = np.percentile(expected_y_batch, 97.5, axis=1)
        pred_sd[start_idx:end_idx]  = np.std(expected_y_batch, axis=1)

        # summary stats for each category probability
        for k in range(K):
            prob_cats_mean[start_idx:end_idx, k] = np.mean(prob_batch[:, :, k], axis=1)
            prob_cats_median[start_idx:end_idx, k] = np.median(prob_batch[:, :, k], axis=1)
            prob_cats_lower[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 2.5, axis=1)
            prob_cats_upper[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 97.5, axis=1)
            prob_cats_sd[start_idx:end_idx, k]  = np.std(prob_batch[:, :, k], axis=1)
        
        all_draws[start_idx:end_idx, :] = expected_y_batch

        # Free batch arrays before next loop
        del logits_batch, prob_batch, expected_y_batch
        

    # Add results summary stats to strat_frame
    strat_frame[f'{belief_col}_pred_mean'] = pred_mean
    strat_frame[f'{belief_col}_pred_median'] = pred_median
    strat_frame[f'{belief_col}_pred_lower'] = pred_lower
    strat_frame[f'{belief_col}_pred_upper'] = pred_upper
    strat_frame[f'{belief_col}_pred_sd'] = pred_sd
    for k in range(K):
        strat_frame[f'{belief_col}_prob_cat_mean_{k+1}'] = prob_cats_mean[:, k]
        strat_frame[f'{belief_col}_prob_cat_median_{k+1}'] = prob_cats_median[:, k]
        strat_frame[f'{belief_col}_prob_cat_lower_{k+1}'] = prob_cats_lower[:, k]
        strat_frame[f'{belief_col}_prob_cat_upper_{k+1}'] = prob_cats_upper[:, k]
        strat_frame[f'{belief_col}_prob_cat_sd_{k+1}'] = prob_cats_sd[:, k]
    
    # save subsampled expected value draws in a DataFrame, and copy weight column and group idx column into the dataframe

    n_star_cols = [c for c in strat_frame.columns if c.startswith('N_star.')]
    n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
    extra_cols = ['state_simple','state_idx','sex_idx','age_idx','eth_idx','income_idx','vote_idx']

    draws_df = pd.concat([
    pd.DataFrame(all_draws, columns=[f'draw_{i+1}' for i in range(n_draws)]),
    strat_frame[n_star_draws + ['N_star.median'] + extra_cols].reset_index(drop=True)
], axis=1)

    mu_draws_df = pd.concat([
    pd.DataFrame(mu_all_draws, columns=[f'mu_draw_{i+1}' for i in range(n_draws)]),
    strat_frame[n_star_draws + ['N_star.median'] + extra_cols].reset_index(drop=True)
], axis=1)
    
    return strat_frame, all_draws, draws_df, K, draw_indices, mu_draws_df

# strat_frame: original stratification frame + prediction's summary stats
# all_draws: 2D array of expected value draws (n_cells, 500)
# draws_df: all_draws in DataFrame + weights + grouping keys (n_cells, 500+)
# K: number of belief categories
# draw_indices: which of the original 8000 draws were selected
# mu_draws_df: category-1 linear predictor draws DataFrame + weights + grouping keys (n_cells, 500+)

In [18]:
# weighted group means and summaries

def aggregate_draws_by_group(draws_df, group_col):
    """
    Aggregate expected value draws by a single group column with posterior weight draws.
    - For each expected-value draw, compute a weighted group average using a randomly
      selected weight draw (N_star.*).
    - Returns one draw column per expected-value draw.
    """
    draw_cols = [c for c in draws_df.columns if c.startswith('draw_')]
    if not draw_cols:
        raise ValueError("No draw_* columns found in draws_df.")

    # Check for posterior weight draws (N_star.1, N_star.2, ..., N_star.104)
    n_star_cols = [c for c in draws_df.columns if c.startswith('N_star.')]
    n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
    if not n_star_draws:
        raise ValueError("No N_star.* weight draws found in draws_df.")
    
    print(f"Using {len(n_star_draws)} posterior weight draws for aggregation")
    all_results = []
        
    weight_indices = np.random.choice(len(n_star_draws), size=len(draw_cols))
    selected_weights = [n_star_draws[idx] for idx in weight_indices]

    for draw_col, weight_col in zip(draw_cols, selected_weights):
        w = draws_df[weight_col].astype(float)
        weighted_draw = draws_df[draw_col] * w
        num = weighted_draw.groupby(draws_df[group_col]).sum()  # sum of weighted draws
        den = w.groupby(draws_df[group_col]).sum()  # sum of weights
        group_result = (num / den).reset_index()
        group_result.columns = [group_col, draw_col]
        group_result.set_index(group_col, inplace=True)
        all_results.append(group_result)
    
    # Concatenate along axis=1 and reset index to get the group column back
    group_draws = pd.concat(all_results, axis=1).reset_index()

    return group_draws

def summarize_group_draws(group_draws_df, group_col):
    draw_cols = [c for c in group_draws_df.columns if c.startswith('draw_')]
    if not draw_cols:
        raise ValueError("No draw_* columns found in group_draws_df.")
    vals = group_draws_df[draw_cols]
    out = pd.DataFrame({group_col: group_draws_df[group_col]})
    out['pred_mean'] = vals.mean(axis=1)
    out['pred_median'] = vals.median(axis=1)
    out['pred_lower'] = vals.quantile(0.025, axis=1)
    out['pred_upper'] = vals.quantile(0.975, axis=1)
    out['pred_sd'] = vals.std(axis=1)
    return out  # (row = n_groups, columns = group_col + 5 summary stats)


In [21]:
import pickle
import pandas as pd
import numpy as np
import os

# compute group-level category probabilities from cell-level predictions
def compute_group_category_probabilities(pred_df, belief_col, group_col):
    """
    Compute group-level category probabilities for a multinomial model.
    - For each weight draw (N_star.*), compute weighted group probabilities
    - Summarize across all weight draws (mean/median/intervals/sd)
    """
    cat_cols = [c for c in pred_df.columns if c.startswith(f"{belief_col}_prob_cat_mean_")]
    if not cat_cols:
        raise ValueError(f"No category probability mean columns found for {belief_col}.")
    K = max(sorted(int(c.split('_')[-1]) for c in cat_cols))

    n_star_cols = [c for c in pred_df.columns if c.startswith('N_star.')]
    n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
    if not n_star_draws:
        raise ValueError("No N_star.* weight draws found in pred_df.")
    
    print(f"Using all {len(n_star_draws)} posterior weight draws for category probability aggregation")
    
    out_rows = []
    for k in range(1, K + 1):
        col_k = f"{belief_col}_prob_cat_mean_{k}"
        if col_k not in pred_df.columns:
            continue
        
        # compute group probabilities for each weight draw
        draw_vals = []
        for w_col in n_star_draws:
            w = pred_df[w_col].astype(float)
            num_by_group = (pred_df[col_k] * w).groupby(pred_df[group_col]).sum()
            denom_by_group = w.groupby(pred_df[group_col]).sum()
            prob_by_group = (num_by_group / denom_by_group)
            draw_vals.append(prob_by_group)
        
        # combine draws: index=group, columns=weight draws
        draw_mat = pd.concat(draw_vals, axis=1)
        draw_mat = draw_mat.sort_index()
        group_index = draw_mat.index
        prob_vals = draw_mat.to_numpy()
        
        # summarize across weight draws
        summary = pd.DataFrame({
            group_col: list(group_index),
            'category': k,
            'prob_mean': prob_vals.mean(axis=1),
            'prob_median': np.median(prob_vals, axis=1),
            'prob_lower': np.quantile(prob_vals, 0.025, axis=1),
            'prob_upper': np.quantile(prob_vals, 0.975, axis=1),
            'prob_sd': prob_vals.std(axis=1)
        })
        out_rows.append(summary)
    
    if not out_rows:
        raise ValueError(f"No results generated during aggregation. Check that belief column '{belief_col}' and weight columns are present in pred_df.")
    
    final_result = pd.concat(out_rows, ignore_index=True)
    
    return final_result  # (row = n_groups * K, columns = group_col, category, prob_*)


belief_col = "Belief_opinion_on_what_should_be_done_with_illegal_immigrants_Symbol"
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_sf.csv")

# Get the time windows from fitted models
base_dir = "/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Nominal_Immigrants"

# Find all time window directories
time_window_dirs = [d for d in os.listdir(base_dir) if d.startswith(belief_col)]
time_windows = sorted([d.replace(f"{belief_col}_", "").replace("_stan_output", "") for d in time_window_dirs])

print(f"Found {len(time_windows)} time windows: {time_windows}\n")

# Group mapping for consolidation
group_map = [
    ('state_simple', 'state'),
    ('state_idx', 'state_idx'),
    ('sex_idx', 'sex'),
    ('age_idx', 'age'),
    ('eth_idx', 'eth'),
    ('income_idx', 'income'),
    ('vote_idx', 'vote'),
]

# Initialize storage for consolidated category probabilities across all windows
consolidated_cat_probs = {name: [] for _, name in group_map}

# Loop over each time window
for idx, window in enumerate(time_windows):
    print(f"[{idx+1}/{len(time_windows)}] Processing predictions for window: {window}")
    
    # Load the fitted model for this window
    model_dir = f"{base_dir}/{belief_col}_{window}_stan_output"
    fit_path = f"{model_dir}/fit.pkl"
    
    if not os.path.exists(fit_path):
        print(f"  Warning: fit.pkl not found for window {window}")
        continue
    
    try:
        with open(fit_path, "rb") as f:
            fit = pickle.load(f)
        
        # Generate cell-level predictions
        strat_frame_copy = strat_frame.copy()  # Use a copy to avoid modifying original
        strat_frame_pred, all_draws, draws_df, K, draw_indices, mu_draws_df = generate_cell_level_predictions(
            fit,
            strat_frame_copy,
            belief_col,
            batch_size=10000,
            n_keep_draws=500
        )
        
        # Save cell-level predictions (summaries)
        strat_frame_pred.to_csv(f"{model_dir}/cell_level_predictions.csv", index=False)
        # Save expected value draws per cell
        draws_df.to_csv(f"{model_dir}/cell_level_all_draws.csv", index=False)

        # Save mu draws per cell 
        if mu_draws_df is not None:
            try:
                mu_draws_df.to_parquet(f"{model_dir}/cell_level_mu_draws.parquet")
            except Exception:
                mu_draws_df.to_csv(f"{model_dir}/cell_level_mu_draws.csv.gz", index=False, compression='gzip')

        # State-level aggregation and save (expected value draws)
        state_group_col = 'state_simple' if 'state_simple' in draws_df.columns else ('state_idx' if 'state_idx' in draws_df.columns else None)
        if state_group_col is not None:
            state_all_draws = aggregate_draws_by_group(draws_df, state_group_col)
            state_preds = summarize_group_draws(state_all_draws, state_group_col)
            state_all_draws.to_csv(f"{model_dir}/state_level_all_draws.csv", index=False)
            state_preds.to_csv(f"{model_dir}/state_level_predictions.csv", index=False)
        else:
            print("Warning: no state column found in draws_df; skipping state-level aggregation.")

        # Demographic group aggregations and save (expected value draws)
        demo_group_map = [
            ('sex_idx', 'sex'),
            ('age_idx', 'age'),
            ('eth_idx', 'eth'),
            ('income_idx', 'income'),
            ('vote_idx', 'vote'),
        ]
        for gcol, gname in demo_group_map:
            if gcol in draws_df.columns:
                grp_draws = aggregate_draws_by_group(draws_df, gcol)
                grp_preds = summarize_group_draws(grp_draws, gcol)
                grp_draws.to_csv(f"{model_dir}/group_{gname}_all_draws.csv", index=False)
                grp_preds.to_csv(f"{model_dir}/group_{gname}_predictions.csv", index=False)
            else:
                print(f"Note: column '{gcol}' not found; skipping group '{gname}'.")
        
        # Compute category probabilities for all groups and store for consolidation
        print(f"  Computing category probabilities using all weight samples...")
        for gcol, gname in group_map:
            if gcol not in strat_frame_pred.columns:
                continue
            try:
                grp_cat = compute_group_category_probabilities(
                    strat_frame_pred, belief_col, gcol
                )
                grp_cat['time_window_start'] = window
                consolidated_cat_probs[gname].append(grp_cat)
            except Exception as e:
                print(f"    Warning: failed to compute {gname} category probs: {e}")
        
        print(f"  Successfully generated all predictions for window {window}\n")
        
    except Exception as e:
        print(f"  Failed to process window {window}: {e}\n")
        continue

print("All time windows processed!")

# Save consolidated category probability timeseries files
print("\nSaving consolidated category probability timeseries...")
for gname, parts in consolidated_cat_probs.items():
    if parts:
        df_all = pd.concat(parts, ignore_index=True)
        output_path = f"{base_dir}/group_{gname}_category_probs_timeseries.csv"
        df_all.to_csv(output_path, index=False)
        print(f" Saved {gname}: {df_all.shape[0]} rows, {df_all['time_window_start'].nunique()} windows")
    else:
        print(f" No data for {gname}")

print("\nPrediction and consolidation pipeline complete!")

Found 26 time windows: ['2023-11-05 00:00:00+00:00', '2023-11-19 00:00:00+00:00', '2023-12-03 00:00:00+00:00', '2023-12-17 00:00:00+00:00', '2023-12-31 00:00:00+00:00', '2024-01-14 00:00:00+00:00', '2024-01-28 00:00:00+00:00', '2024-02-11 00:00:00+00:00', '2024-02-25 00:00:00+00:00', '2024-03-10 00:00:00+00:00', '2024-03-24 00:00:00+00:00', '2024-04-07 00:00:00+00:00', '2024-04-21 00:00:00+00:00', '2024-05-05 00:00:00+00:00', '2024-05-19 00:00:00+00:00', '2024-06-02 00:00:00+00:00', '2024-06-16 00:00:00+00:00', '2024-06-30 00:00:00+00:00', '2024-07-14 00:00:00+00:00', '2024-07-28 00:00:00+00:00', '2024-08-11 00:00:00+00:00', '2024-08-25 00:00:00+00:00', '2024-09-08 00:00:00+00:00', '2024-09-22 00:00:00+00:00', '2024-10-06 00:00:00+00:00', '2024-10-20 00:00:00+00:00']

[1/26] Processing predictions for window: 2023-11-05 00:00:00+00:00
Total draws available: 800, selected 500 for subsampling
Subsampling draws: using 500 out of 800 posterior draws

Generating predictions for 110,160 cell